# Config G — First REAL LLM Memory-Policy Experiment (Kaggle GPU)

**Controlled pilot experiment.** Not publication-level evidence.

**Research question:** *Can a learned LLM memory policy learn when to STORE, RETRIEVE, UPDATE, SUMMARIZE, DISCARD, or NOOP, and does this improve memory-conditioned agent performance?*

This notebook runs the **first real** LoRA/QLoRA GRPO training of the Config-G LLM memory policy and reports honest before/after numbers. It uses the existing repository implementation (`app.memory.trajectory`, `app.memory.llm_policy`, `app.memory.rl.llm_grpo`, `app.memory.metrics_suite`) — nothing is reimplemented. **No result is fabricated, simulated, or extrapolated.** If a GPU, the model, or dependencies are unavailable, the notebook STOPS and records the blocker.

Setup steps are in the final **Reproducibility** section.

## 1. Environment check
Detect hardware + CUDA. **No GPU ⇒ STOP (BLOCKED), no simulated training.**

In [ ]:
import os, sys, platform, subprocess, json, time, datetime

REPORT = {'environment': {}, 'blockers': []}
def sh(cmd):
    try: return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e: return f'(failed: {e})'

env = REPORT['environment']
env['python'] = sys.version.split()[0]
env['os'] = platform.platform()
env['arch'] = platform.machine()
env['cpu_cores'] = os.cpu_count()
env['ram'] = sh("free -h | awk '/Mem:/{print $2}'") or 'unknown'
env['disk_free'] = sh("df -h /kaggle/working 2>/dev/null | tail -1 | awk '{print $4}'") or sh("df -h / | tail -1 | awk '{print $4}'")
print('Python   :', env['python'])
print('OS       :', env['os'])
print('Arch     :', env['arch'])
print('CPU cores:', env['cpu_cores'])
print('RAM      :', env['ram'])
print('Disk free:', env['disk_free'])
print('-'*50)
print(sh('nvidia-smi') or 'nvidia-smi: not found')
print('-'*50)

gpu_ok = False
try:
    import torch
    gpu_ok = torch.cuda.is_available()
    env['torch'] = torch.__version__
    env['cuda_available'] = gpu_ok
    env['cuda_version'] = getattr(torch.version, 'cuda', None)
    if gpu_ok:
        env['gpu_name'] = torch.cuda.get_device_name(0)
        env['gpu_vram_gb'] = round(torch.cuda.get_device_properties(0).total_memory/1e9, 2)
        print('GPU      :', env['gpu_name'], f"({env['gpu_vram_gb']} GB VRAM)")
        print('CUDA     :', env['cuda_version'], '| torch', env['torch'])
except Exception as e:
    env['torch'] = f'import failed: {e}'

if not gpu_ok:
    REPORT['status'] = 'BLOCKED_NOT_RUN'
    REPORT['blockers'].append({'id':'no_gpu','evidence':'torch.cuda.is_available() is False / nvidia-smi absent',
        'impact':'No CUDA accelerator; refusing to run CPU-simulated training.'})
    print('\n=== STATUS: BLOCKED_NOT_RUN (no GPU) ===')
    raise SystemExit('No GPU detected — stopping. Enable Kaggle GPU (Settings -> Accelerator -> GPU T4).')
print('\nGPU present — continuing.')

## 2. Clone repo + install dependencies
Installs the finetune stack and puts the repo on `sys.path` (avoids the `requires-python>=3.11` editable-install issue on Kaggle).

In [ ]:
REPO_URL = 'https://github.com/zda25m005-netizen/agentic-ai-os.git'  # <- change if you use a fork/private mirror
REPO_DIR = '/kaggle/working/agentic-ai-os'

# Always fetch the LATEST main so pushed fixes are picked up on re-run.
if not os.path.isdir(REPO_DIR):
    print(sh(f'git clone --depth 1 {REPO_URL} {REPO_DIR}'))
else:
    print(sh(f'cd {REPO_DIR} && git fetch --depth 1 origin main && git reset --hard origin/main'))
print('HEAD:', sh(f'cd {REPO_DIR} && git log --oneline -1'))
assert os.path.isdir(os.path.join(REPO_DIR,'app','memory')), 'repo clone failed or wrong URL'

# Kaggle already ships torch/transformers/accelerate (CUDA). Add peft; try bitsandbytes for QLoRA.
print(sh('pip install -q -U peft accelerate transformers'))
bnb_out = sh('pip install -q bitsandbytes')
print(bnb_out or 'bitsandbytes install attempted')

# Best-effort editable install WITHOUT deps, so Kaggle's CUDA torch is never downgraded.
# We import the repo via sys.path regardless, so this failing is harmless.
editable = sh(f'cd {REPO_DIR} && pip install -q -e . --no-deps 2>&1 | tail -2')
print('editable install (no-deps, best-effort):', editable or '(ok)')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

vers = {}
import importlib
for m in ('torch','transformers','peft','accelerate'):
    try: vers[m] = importlib.import_module(m).__version__
    except Exception as e: vers[m] = f'MISSING ({e})'
QUANTIZED = False
try:
    import bitsandbytes as bnb
    vers['bitsandbytes'] = bnb.__version__
    QUANTIZED = True  # QLoRA available
except Exception as e:
    vers['bitsandbytes'] = f'unavailable ({e})'
    QUANTIZED = False  # will use plain LoRA, recorded as quantization: none
REPORT['environment']['package_versions'] = vers
REPORT['environment']['quantization'] = 'qlora_4bit' if QUANTIZED else 'none'
print(json.dumps(vers, indent=2))
print('QLoRA available:', QUANTIZED, '=> quantization =', REPORT['environment']['quantization'])
for m in ('torch','transformers','peft','accelerate'):
    assert 'MISSING' not in vers[m], f'{m} failed to import'

## 3. Hugging Face authentication
Reads `HF_TOKEN` from a Kaggle secret (or env). Never printed. Verifies access to the exact model; **401/403 ⇒ STOP**, no substitution.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
print('HF_TOKEN found:', bool(HF_TOKEN))  # never print the token itself

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    try:
        from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e:
        print('login warning:', e)

# Verify access to the EXACT model. Stop on 401/403.
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError
try:
    info = HfApi().model_info(MODEL_ID, token=HF_TOKEN)
    print('Model access OK:', info.id, '| gated:', getattr(info,'gated',None))
except Exception as e:
    code = getattr(getattr(e,'response',None),'status_code',None)
    REPORT['status'] = 'BLOCKED_NOT_RUN'
    REPORT['blockers'].append({'id':'model_access','evidence':f'{type(e).__name__}: {e} (status={code})',
        'impact':'Cannot access the configured model; not substituting another model.'})
    print('\n=== STATUS: BLOCKED_NOT_RUN (model access) ===')
    raise SystemExit(f'Model access failed for {MODEL_ID}: {e}')

## 4–5. Load Config-G code + the controlled pilot configuration
Same env used for baseline **and** trained eval (fair comparison).

In [ ]:
from app.memory.trajectory import TrajectoryMemoryEnv, MemoryOp
from app.memory.llm_policy import LLMMemoryPolicy, HeuristicPolicy
from app.memory.rl.llm_grpo import LLMGRPOTrainer
from app.memory.metrics_suite import evaluate_policy_on_env, policy_action_accuracy, latency_stats
import numpy as np

SEED = 0
env = TrajectoryMemoryEnv(n=100, seed=SEED)        # the ONE evaluation env (shared)
STATES = env.states()
OPTIMAL = [int(s['optimal']) for s in STATES]

CONFIG = {
    'model_id': MODEL_ID,
    'dataset': 'TrajectoryMemoryEnv(n=100, seed=0)',
    'decision_steps': len(STATES),
    'seed': SEED,
    'lora_r': 8, 'lora_alpha': 16, 'target_modules': ['q_proj','v_proj'],
    'learning_rate': 1e-4, 'optimizer': 'AdamW', 'grpo_group_size': 8,
    'epochs': 1, 'max_states': 64, 'planned_optimizer_steps': 64,
    'quantization': REPORT['environment']['quantization'],
}
print(json.dumps(CONFIG, indent=2))
print('decision steps in env:', len(STATES))

### Device-correct overrides (robust)
Replaces the Config-G trainer/policy methods with CUDA-correct versions at runtime, so the experiment does not depend on whether the GPU fix is in the cloned files. Same logic as the repo fix.

In [ ]:
# --- Config-G device-correct overrides (robust; layout-independent) --------------
# Replaces the trainer/policy methods with CUDA-correct versions at runtime, so the run
# does not depend on whether the GPU fix is present in the cloned files. Same logic as the
# repo fix; guarantees all tensors live on the model's device.
import numpy as _np, torch as _torch
from app.memory.rl import llm_grpo as _grpo
from app.memory import llm_policy as _pol
from app.memory.trajectory.actions import OPS as _OPS, MemoryOp as _Op
from app.memory.trajectory.prompt import build_prompt as _bp, parse_action as _pa
from app.memory.trajectory.env import TrajectoryMemoryEnv as _Env

def _build_model(self):
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    on_gpu = _torch.cuda.is_available()
    tok = AutoTokenizer.from_pretrained(self.model_name)
    kwargs = {"torch_dtype": _torch.float16 if on_gpu else _torch.float32}
    if self.quantized:
        from transformers import BitsAndBytesConfig
        kwargs = {"quantization_config": BitsAndBytesConfig(load_in_4bit=True,
                  bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=_torch.float16),
                  "device_map": "auto"}
    model = AutoModelForCausalLM.from_pretrained(self.model_name, **kwargs)
    if self.quantized:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)
    elif on_gpu:
        model = model.to("cuda")
    lora = LoraConfig(r=self.lora_r, lora_alpha=self.lora_alpha,
                      target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")
    return tok, get_peft_model(model, lora)

def _train(self, env=None, *, epochs=1, max_states=None, out_dir=None):
    if not _grpo.backend_available():
        raise RuntimeError("LLM GRPO training needs torch+transformers+peft (.[finetune]).")
    env = env or _Env(seed=self.seed)
    tok, model = self._build_model()
    model.train()
    device = next(model.parameters()).device
    opt = _torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=self.lr)
    rng = _np.random.default_rng(self.seed)
    action_ids = [tok(a, add_special_tokens=False)["input_ids"][0] for a in _OPS]
    states = env.states()
    if max_states: states = states[:max_states]
    history = []
    for _ in range(epochs):
        for s in states:
            prompt = tok.apply_chat_template(_bp(s), tokenize=False, add_generation_prompt=True)
            inputs = tok(prompt, return_tensors="pt").to(device)
            logits = model(**inputs).logits[0, -1]
            logp = _torch.log_softmax(logits, dim=-1)
            act_logp = _torch.stack([logp[i] for i in action_ids])
            probs = _torch.softmax(act_logp.detach(), dim=-1).float().cpu().numpy()
            sampled = rng.choice(len(_OPS), size=self.group_size, p=probs/probs.sum())
            rewards = [env.reward(s, _Op(int(a))) for a in sampled]
            adv = _grpo.group_advantages(rewards)
            loss = -_torch.stack([act_logp[int(a)]*float(A)
                                  for a,A in zip(sampled, adv)]).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        history.append({"loss": float(loss.item())})
    if out_dir:
        model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return {"epochs": epochs, "states": len(states), "adapter_dir": out_dir, "history": history}

_grpo.LLMGRPOTrainer._build_model = _build_model
_grpo.LLMGRPOTrainer.train = _train

def _ensure_model(self):
    if self._model is not None: return True
    if not _pol.backend_available(): return False
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        on_gpu = _torch.cuda.is_available()
        tok = AutoTokenizer.from_pretrained(self.model_name)
        model = AutoModelForCausalLM.from_pretrained(
            self.model_name, torch_dtype=_torch.float16 if on_gpu else _torch.float32)
        if self.adapter_path:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, self.adapter_path)
        if on_gpu: model = model.to("cuda")
        model.eval(); self._tokenizer, self._model = tok, model
        return True
    except Exception as e:
        print("model load failed:", e); return False

def _act(self, state):
    if not self._ensure_model(): return self.fallback.act(state)
    try:
        prompt = self._tokenizer.apply_chat_template(_bp(state), tokenize=False, add_generation_prompt=True)
        inputs = self._tokenizer(prompt, return_tensors="pt").to(self._model.device)
        with _torch.no_grad():
            out = self._model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        text = self._tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return _pa(text)
    except Exception as e:
        print("act failed:", e); return self.fallback.act(state)

_pol.LLMMemoryPolicy._ensure_model = _ensure_model
_pol.LLMMemoryPolicy.act = _act
print("Config-G device-correct overrides installed | CUDA:", _torch.cuda.is_available())

## 6–7. Baseline evaluation (A random, B heuristic, C base LLM)
Metrics reported are those the existing code genuinely supports on this env: **policy_action_accuracy** (overall + per-op) and **decision_latency**. Label-derived confusion rates (unnecessary write/retrieval, stale-leak) are computed from the env's optimal labels. Metrics that need a downstream-task harness (retrieval P/R/F1, temporal-update, task success, token efficiency) are recorded as `not_measured` with a reason — never faked.

In [ ]:
def label_derived_rates(pred, opt):
    STORE,RETRIEVE,UPDATE,SUMMARIZE,DISCARD,NOOP = 0,1,2,3,4,5
    n=len(opt)
    unnec_write = sum(1 for p,o in zip(pred,opt) if p==STORE and o in (NOOP,DISCARD))/n
    unnec_ret   = sum(1 for p,o in zip(pred,opt) if p==RETRIEVE and o!=RETRIEVE)/n
    stale_leak  = sum(1 for p,o in zip(pred,opt) if o==UPDATE and p in (NOOP,STORE))/max(1,sum(1 for o in opt if o==UPDATE))
    return {'unnecessary_write_rate':round(unnec_write,4),'unnecessary_retrieval_rate':round(unnec_ret,4),
            'stale_memory_leak_rate':round(stale_leak,4),'_scope':'label_derived_from_env_optimal'}

NOT_MEASURED = {k:'not_measured' for k in ['memory_retrieval_precision','memory_retrieval_recall',
    'memory_retrieval_f1','temporal_update_accuracy','task_success','token_efficiency','memory_induced_error_rate']}
NOT_MEASURED_REASON = ('requires a downstream-task / retrieved-vs-gold harness that this trajectory env does '
    'not produce; recorded honestly rather than fabricated')

def eval_policy(pred, latencies=None):
    paa = policy_action_accuracy(pred, OPTIMAL)
    out = {'policy_action_accuracy': paa['accuracy'], 'per_op': paa['per_op'], 'n': paa['n']}
    out.update(label_derived_rates(pred, OPTIMAL))
    out.update(NOT_MEASURED)
    if latencies is not None: out['decision_latency'] = latency_stats(latencies)
    return out

# A. random reference (seeded)
rng = np.random.default_rng(SEED)
rand_pred = [int(rng.integers(6)) for _ in STATES]
res_random = eval_policy(rand_pred)

# B. heuristic policy
heur = HeuristicPolicy()
lat=[]; hp=[]
for s in STATES:
    t=time.perf_counter(); a=heur.act(s); lat.append((time.perf_counter()-t)*1000); hp.append(int(a))
res_heuristic = eval_policy(hp, lat)

# C. base LLM (no adapter) — real generations on GPU
base_pol = LLMMemoryPolicy(MODEL_ID)
assert base_pol.available(), 'LLM backend not available'
lat=[]; bp=[]
t0=time.perf_counter()
for i,s in enumerate(STATES):
    t=time.perf_counter(); a=base_pol.act(s); lat.append((time.perf_counter()-t)*1000); bp.append(int(a))
    if i%50==0: print(f'  base LLM {i}/{len(STATES)}')
res_base = eval_policy(bp, lat)
print('base LLM eval took %.1fs'%(time.perf_counter()-t0))
print('random   PAA:', res_random['policy_action_accuracy'])
print('heuristic PAA:', res_heuristic['policy_action_accuracy'])
print('base LLM  PAA:', res_base['policy_action_accuracy'])

## 8. Training — one REAL LoRA/QLoRA GRPO run
Uses `LLMGRPOTrainer` from the repo. Saves the adapter. Logs full provenance.

In [ ]:
ADAPTER_DIR = os.path.join(REPO_DIR, 'experiments/config_g_lora/adapter')
os.makedirs(ADAPTER_DIR, exist_ok=True)

trainer = LLMGRPOTrainer(MODEL_ID, group_size=CONFIG['grpo_group_size'], lr=CONFIG['learning_rate'],
                         lora_r=CONFIG['lora_r'], lora_alpha=CONFIG['lora_alpha'],
                         quantized=QUANTIZED, seed=SEED)
start = datetime.datetime.now(datetime.timezone.utc).isoformat(); t0=time.perf_counter()
train_report = trainer.train(env, epochs=CONFIG['epochs'], max_states=CONFIG['max_states'], out_dir=ADAPTER_DIR)
dur = time.perf_counter()-t0; end = datetime.datetime.now(datetime.timezone.utc).isoformat()

REPORT['training'] = {**CONFIG, 'quantization': REPORT['environment']['quantization'],
    'optimizer_steps_actual': train_report.get('states'), 'epochs_actual': train_report.get('epochs'),
    'final_loss': (train_report.get('history') or [{}])[-1].get('loss'),
    'start_utc': start, 'end_utc': end, 'duration_sec': round(dur,2),
    'adapter_dir': train_report.get('adapter_dir')}
print('training done in %.1fs'%dur, '| final loss:', REPORT['training']['final_loss'])
print('adapter saved ->', ADAPTER_DIR)

## 9. Trained-policy evaluation (D) — SAME env
Reloads the base model + trained adapter and evaluates on the identical `env`.

In [ ]:
trained_pol = LLMMemoryPolicy(MODEL_ID, adapter_path=ADAPTER_DIR)
assert trained_pol.available()
lat=[]; tp=[]
for i,s in enumerate(STATES):
    t=time.perf_counter(); a=trained_pol.act(s); lat.append((time.perf_counter()-t)*1000); tp.append(int(a))
    if i%50==0: print(f'  trained LLM {i}/{len(STATES)}')
res_trained = eval_policy(tp, lat)
print('trained LLM PAA:', res_trained['policy_action_accuracy'])

## 10. Results JSON + BEFORE → TRAIN → AFTER report

In [ ]:
delta = {
  'policy_action_accuracy_base_to_trained': round(res_trained['policy_action_accuracy'] - res_base['policy_action_accuracy'], 4),
  'policy_action_accuracy_vs_heuristic': round(res_trained['policy_action_accuracy'] - res_heuristic['policy_action_accuracy'], 4),
}
REPORT.update({
  'experiment': 'config_G_llm_memory_policy_lora_grpo',
  'status': 'COMPLETED',
  'kind': 'controlled_pilot_experiment',
  'research_question': ('Can a learned LLM memory policy learn when to STORE/RETRIEVE/UPDATE/'
      'SUMMARIZE/DISCARD/NOOP, and does this improve memory-conditioned agent performance?'),
  'model': {'model_id': MODEL_ID, 'quantization': REPORT['environment']['quantization']},
  'hardware': REPORT['environment'],
  'evaluation': {'random': res_random, 'heuristic': res_heuristic, 'base_llm': res_base, 'trained_llm': res_trained},
  'delta': delta,
  'checkpoint': {'adapter_dir': ADAPTER_DIR, 'committed_to_git': False,
      'note': 'LoRA adapter weights are NOT committed; regenerate via this notebook.'},
  'not_measured_reason': NOT_MEASURED_REASON,
  'limitations': [
    'Controlled pilot on a synthetic labelled trajectory env (n=100, 401 steps) — not publication-level evidence.',
    'Only policy_action_accuracy + decision_latency are directly supported metrics; retrieval P/R/F1, '
    'temporal-update, task success, and token efficiency need a downstream-task harness (not_measured).',
    'Single seed, single epoch, 64 optimizer steps.',
  ],
})

OUT = os.path.join(REPO_DIR, 'experiments/config_g_lora/results.json')
with open(OUT,'w') as f: json.dump(REPORT, f, indent=2, default=str)
with open('/kaggle/working/results.json','w') as f: json.dump(REPORT, f, indent=2, default=str)
print('wrote', OUT)

def paa(r): return r['policy_action_accuracy']
print('\n'+'='*40)
print('CONFIG G REAL EXPERIMENT')
print('='*40)
print('ENVIRONMENT'); print('  GPU:', REPORT['environment'].get('gpu_name'), '| CUDA', REPORT['environment'].get('cuda_version'),
      '| quant', REPORT['environment']['quantization'])
print('BASELINE (policy_action_accuracy)')
print('  A random   :', paa(res_random))
print('  B heuristic:', paa(res_heuristic))
print('  C base LLM :', paa(res_base))
print('TRAINING'); print('  steps:', REPORT['training']['optimizer_steps_actual'], '| loss:', REPORT['training']['final_loss'],
      '| %.1fs'%REPORT['training']['duration_sec'])
print('TRAINED POLICY'); print('  D trained  :', paa(res_trained))
print('DELTA'); print('  base->trained:', delta['policy_action_accuracy_base_to_trained'])
print('  trained-heuristic:', delta['policy_action_accuracy_vs_heuristic'])
print('CHECKPOINT'); print('  ', ADAPTER_DIR)
print('STATUS'); print('  ', REPORT['status'])
print('='*40)

## 11. Reproducibility

1. **Enable GPU:** Kaggle → notebook → **Settings ▸ Accelerator ▸ GPU T4 x1** (or P100).
2. **Add HF token:** **Add-ons ▸ Secrets ▸ Add secret**, name it exactly `HF_TOKEN`, paste a read token from https://huggingface.co/settings/tokens. (Qwen2.5-0.5B is public; the token is verified but usually not strictly required.)
3. **Run:** `Run All`. The notebook clones the repo, installs deps, trains, and evaluates.
4. **results.json:** written to `agentic-ai-os/experiments/config_g_lora/results.json` and `/kaggle/working/results.json` (download this).
5. **LoRA adapter:** saved to `agentic-ai-os/experiments/config_g_lora/adapter/` (ephemeral Kaggle output; not committed to git).
6. **Exact config:** Qwen/Qwen2.5-0.5B-Instruct · LoRA r=8 α=16 (q_proj,v_proj) · AdamW lr=1e-4 · GRPO group=8 · epochs=1 · max_states=64 · seed=0 · env `TrajectoryMemoryEnv(n=100, seed=0)` (401 steps).
7. **Reproduce:** same seed + same env ⇒ same baseline; training is seeded (`seed=0`). Report GPU model alongside numbers.

**Send back:** the printed `CONFIG G REAL EXPERIMENT` block **and** the downloaded `results.json`.